# Métricas de PrimeKG integrado con la capa de drogas

El grafo cargado en `neo4j-thesis` es PrimeKG integrado (genes, enfermedades y pathways) más los compuestos de la red del laboratorio, unidos por `grafo_laboratorio/armar_grafo.py` y cargados por `neo4j/cargar_grafo_laboratorio.py`. Este notebook lee el grafo de Neo4j, no de los CSV, y calcula las métricas básicas: cuántos nodos y aristas hay por capa, cuántas aristas hay dentro de cada capa y entre cada par de capas, cómo son los planos homogéneos, y cómo se ven tres enfermedades de ejemplo (una psiquiátrica, la celiaquía y la gripe). Cada bloque termina con una lectura, y al final se razona si la carga funcionó.

Para que la comparación tenga sentido, los conteos de Neo4j se contrastan contra los archivos de los que salió la carga: `datos/processed/merged_nodes.csv` y `merged_edges.csv` para PrimeKG integrado, y `grafo_laboratorio/grafo/` para los compuestos.

In [1]:
import os
import sys
from collections import Counter, defaultdict

import networkx as nx
import numpy as np
import pandas as pd

sys.path.insert(0, "neo4j")
import config  # noqa: E402
from neo4j import GraphDatabase

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 90)
SEED = 42

driver = GraphDatabase.driver(config.NEO4J_URI, auth=(config.NEO4J_USER, config.NEO4J_PASSWORD))


def consulta(cypher, **parametros):
    with driver.session() as session:
        return pd.DataFrame([dict(fila) for fila in session.run(cypher, parametros)])


#todos los nodos con las propiedades que hacen falta para separar subtipos
nodos = consulta("""MATCH (n) RETURN labels(n)[0] AS label, n.id AS id, n.name AS name,
                    coalesce(n.es_grupo_bert, false) AS es_grupo_bert, coalesce(n.es_complejo_signor, false) AS es_complejo_signor""")
#todas las aristas, una por relacion (cada relacion esta guardada en un solo sentido)
aristas = consulta("""MATCH (a)-[r]->(b) RETURN type(r) AS tipo, labels(a)[0] AS label_a, a.id AS id_a,
                      labels(b)[0] AS label_b, b.id AS id_b, r.weight AS weight""")
print(len(nodos), "nodos y", len(aristas), "aristas leidos de Neo4j")

119706 nodos y 468052 aristas leidos de Neo4j


## 1. Nodos: Neo4j contra los archivos de origen

Se separan los subtipos que conviven bajo un mismo label: los complejos de SIGNOR dentro de `Gene` y los grupos BERT dentro de `Disease`.

In [2]:
def subtipo(fila):
    if fila.label == "Gene":
        return "complejo SIGNOR" if fila.es_complejo_signor else "gen"
    if fila.label == "Disease":
        return "grupo BERT" if fila.es_grupo_bert else "CUI"
    return fila.label.lower()


nodos["subtipo"] = nodos.apply(subtipo, axis=1)
en_neo4j = nodos.groupby(["label", "subtipo"]).size().rename("neo4j")

origen = pd.read_csv("datos/processed/merged_nodes.csv", dtype={"node_id": str})
etiqueta = {"gene_protein": "Gene", "disease": "Disease", "pathway": "Pathway"}
origen["label"] = origen.node_type.map(etiqueta)
origen["subtipo"] = np.select(
    [origen.node_id.str.startswith("SIGNOR"), origen.label == "Gene", (origen.label == "Disease") & (origen.node_source == "primekg"), origen.label == "Disease"],
    ["complejo SIGNOR", "gen", "grupo BERT", "CUI"], default="pathway")
nodos_lab = pd.read_csv(config.ARCHIVO_LAB_NODOS, dtype=str, usecols=["node_type"])
drogas_csv = nodos_lab[nodos_lab.node_type == "drug"]
en_archivos = pd.concat([origen.groupby(["label", "subtipo"]).size(),
                         pd.Series({("Drug", "drug"): len(drogas_csv)})]).rename("archivos")

tabla_nodos = pd.concat([en_neo4j, en_archivos], axis=1).fillna(0).astype(int)
tabla_nodos["coincide"] = tabla_nodos.neo4j == tabla_nodos.archivos
print("ids repetidos por label en Neo4j:", int(nodos.duplicated(["label", "id"]).sum()))
tabla_nodos

ids repetidos por label en Neo4j: 0


neo4j  archivos  coincide
Disease CUI              15039     15039      True
        grupo BERT        1040      1040      True
Drug    drug             83867     83867      True
Gene    complejo SIGNOR    421       421      True
        gen              17322     17322      True
Pathway pathway           2017      2017      True

**Lectura.** Los 119.706 nodos están, con los mismos subtipos que en los archivos, y ningún id se repite dentro de un label. La carga no perdió ni duplicó nodos. Lo de PrimeKG integrado queda igual que en su origen (35.839 nodos) y la capa `Drug` suma 83.867 compuestos.

## 2. Aristas por tipo: Neo4j contra los archivos de origen

`merged_edges.csv` guarda cada arista en los dos sentidos, así que se cuentan los pares distintos sin orden. La estrella de los grupos BERT viene dentro de `disease_disease` y el loader la separa en `DISEASE_BERT` con `disease_bert_edges.csv`; acá se hace la misma separación para comparar.

In [3]:
aristas_origen = pd.read_csv("datos/processed/merged_edges.csv", usecols=["x_index", "y_index", "edge_type"])
aristas_origen["par"] = [tuple(sorted(p)) for p in zip(aristas_origen.x_index, aristas_origen.y_index)]
aristas_origen = aristas_origen.drop_duplicates("par")
estrella = pd.read_csv("datos/processed/disease_bert_edges.csv", usecols=["x_index", "y_index"])
pares_estrella = {tuple(sorted(p)) for p in zip(estrella.x_index, estrella.y_index)}
aristas_origen["tipo"] = aristas_origen.edge_type.map(config.RELACION_POR_TIPO)
aristas_origen.loc[aristas_origen.par.isin(pares_estrella), "tipo"] = config.RELACION_ESTRELLA

conteo_archivos = aristas_origen.tipo.value_counts()
aristas_lab = pd.read_csv(config.ARCHIVO_LAB_ARISTAS, dtype=str, usecols=["edge_type"])
for tipo, edge_type in [("INDICATION", "drug_disease"), ("DRUG_TARGET", "drug_gene"), ("DRUG_DRUG", "drug_drug")]:
    conteo_archivos[tipo] = int((aristas_lab.edge_type == edge_type).sum())

tabla_tipos = pd.concat([aristas.tipo.value_counts().rename("neo4j"), conteo_archivos.rename("archivos")], axis=1).fillna(0).astype(int)
tabla_tipos["coincide"] = tabla_tipos.neo4j == tabla_tipos.archivos
#una relacion repetida entre el mismo par, o el mismo par unido en los dos sentidos, seria un error de carga
pares_sin_orden = [tuple(sorted(p)) for p in zip(aristas.label_a + ":" + aristas.id_a, aristas.label_b + ":" + aristas.id_b)]
repetidas = pd.Series(list(zip(aristas.tipo, pares_sin_orden))).duplicated().sum()
bucles = int(((aristas.label_a == aristas.label_b) & (aristas.id_a == aristas.id_b)).sum())
print("relaciones repetidas (mismo tipo y mismo par, en cualquier sentido):", int(repetidas), "| bucles:", bucles)
tabla_tipos.loc["total"] = [tabla_tipos.neo4j.sum(), tabla_tipos.archivos.sum(), tabla_tipos.neo4j.sum() == tabla_tipos.archivos.sum()]
tabla_tipos

relaciones repetidas (mismo tipo y mismo par, en cualquier sentido): 0 | bucles: 0


,neo4j,archivos,coincide
tipo,,,
DRUG_TARGET,131973,131973,True
PPI,110051,110051,True
GDA,84014,84014,True
DRUG_DRUG,72557,72557,True
PATHWAY_PROTEIN,42643,42643,True
DISEASE_DISEASE,11456,11456,True
INDICATION,8496,8496,True
DISEASE_BERT,4977,4977,True
FORM_COMPLEX,1885,1885,True


**Lectura.** Los nueve tipos de relación tienen exactamente la cantidad de pares de su archivo de origen, 468.052 en total, sin relaciones repetidas ni bucles. La separación entre `DISEASE_DISEASE` y la estrella `DISEASE_BERT` da lo mismo que en los archivos.

## 3. Aristas dentro de cada capa y entre capas

La matriz cuenta las aristas por par de labels, sin importar el sentido en que están guardadas. La diagonal son las aristas intracapa y el resto las intercapa. Los ceros también importan: son combinaciones que el diseño no conecta y que no deberían tener ninguna arista.

In [4]:
CAPAS = ["Gene", "Disease", "Drug", "Pathway"]
par_de_capas = [tuple(sorted(p, key=CAPAS.index)) for p in zip(aristas.label_a, aristas.label_b)]
aristas["capa_a"] = [p[0] for p in par_de_capas]
aristas["capa_b"] = [p[1] for p in par_de_capas]

matriz = pd.DataFrame(0, index=CAPAS, columns=CAPAS)
for (a, b), n in Counter(zip(aristas.capa_a, aristas.capa_b)).items():
    matriz.loc[a, b] = n
    matriz.loc[b, a] = n
matriz

,Gene,Disease,Drug,Pathway
Gene,111936,84014,131973,42643
Disease,84014,16433,8496,0
Drug,131973,8496,72557,0
Pathway,42643,0,0,0


In [5]:
#el detalle por tipo de relacion dentro de cada combinacion, incluidas las que tienen cero
combinaciones = [(a, b) for i, a in enumerate(CAPAS) for b in CAPAS[i:]]
detalle = aristas.groupby(["capa_a", "capa_b", "tipo"]).size().rename("aristas").reset_index()
filas = []
for a, b in combinaciones:
    tipos = detalle[(detalle.capa_a == a) & (detalle.capa_b == b)]
    clase = "intracapa" if a == b else "intercapa"
    if tipos.empty:
        filas.append({"clase": clase, "capas": f"{a} - {b}", "tipo": "(ninguna)", "aristas": 0})
    for fila in tipos.itertuples():
        filas.append({"clase": clase, "capas": f"{a} - {b}", "tipo": fila.tipo, "aristas": fila.aristas})
tabla_combinaciones = pd.DataFrame(filas).sort_values(["clase", "capas"], ascending=[False, True]).reset_index(drop=True)
print("cada tipo de relacion une un solo par de capas:", bool(detalle.groupby("tipo").size().eq(1).all()))
tabla_combinaciones

cada tipo de relacion une un solo par de capas: True


,clase,capas,tipo,aristas
0,intracapa,Disease - Disease,DISEASE_BERT,4977
1,intracapa,Disease - Disease,DISEASE_DISEASE,11456
2,intracapa,Drug - Drug,DRUG_DRUG,72557
3,intracapa,Gene - Gene,FORM_COMPLEX,1885
4,intracapa,Gene - Gene,PPI,110051
5,intracapa,Pathway - Pathway,(ninguna),0
6,intercapa,Disease - Drug,INDICATION,8496
7,intercapa,Disease - Pathway,(ninguna),0
8,intercapa,Drug - Pathway,(ninguna),0
9,intercapa,Gene - Disease,GDA,84014


**Lectura.** Cada relación une siempre el mismo par de capas, así que ninguna arista quedó colgada del label equivocado. Las cuatro combinaciones en cero (Pathway con Pathway, con Disease y con Drug) son las que el diseño no conecta: los pathways solo tocan genes. Gene con Gene suma el interactoma y la pertenencia a complejos; Disease con Disease suma MONDO y la estrella de los grupos. La relación más numerosa del grafo es `DRUG_TARGET` (131.973), por encima del interactoma.

## 4. Los planos homogéneos

Cada relación intracapa arma un plano sobre el que puede correr DIAMOnD. Para cada uno: nodos con al menos una arista (y qué fracción de la capa son), aristas, componentes conexas, tamaño de la componente mayor (LCC) y kappa, la media del grado al cuadrado sobre la media del grado, que por encima de 2 indica que hay componente gigante. El plano de enfermedades se mide con y sin los grupos BERT. El de drogas va entero: las aristas `DRUG_DRUG` del laboratorio arrancan en una similaridad de 0,81, así que no hay corte que elegir por debajo.

In [6]:
print("similaridad minima en DRUG_DRUG:", aristas[aristas.tipo == "DRUG_DRUG"].weight.min())
tamano_capa = {"Gene": int(((nodos.label == "Gene") & ~nodos.es_complejo_signor).sum()),
               "Disease": int(((nodos.label == "Disease") & ~nodos.es_grupo_bert).sum()),
               "Drug": int((nodos.label == "Drug").sum())}
grupos_bert = set(nodos[nodos.es_grupo_bert].id)


def metricas_plano(nombre, capa, pares):
    grafo = nx.Graph()
    grafo.add_edges_from(pares)
    if grafo.number_of_nodes() == 0:
        return {"plano": nombre}
    grados = np.array([g for _, g in grafo.degree()])
    componentes = sorted((len(c) for c in nx.connected_components(grafo)), reverse=True)
    return {"plano": nombre, "nodos con arista": grafo.number_of_nodes(),
            "fraccion de la capa": round(grafo.number_of_nodes() / tamano_capa[capa], 3),
            "aristas": grafo.number_of_edges(), "componentes": len(componentes), "LCC": componentes[0],
            "LCC / nodos": round(componentes[0] / grafo.number_of_nodes(), 3),
            "kappa": round(float((grados ** 2).mean() / grados.mean()), 1), "grado mediano": float(np.median(grados))}


def pares(tipo, sin_grupos=False, peso_minimo=None):
    sub = aristas[aristas.tipo == tipo]
    if sin_grupos:
        sub = sub[~sub.id_a.isin(grupos_bert) & ~sub.id_b.isin(grupos_bert)]
    if peso_minimo is not None:
        sub = sub[sub.weight >= peso_minimo]
    return list(zip(sub.id_a, sub.id_b))


planos = pd.DataFrame([
    metricas_plano("gene: PPI", "Gene", pares("PPI")),
    metricas_plano("disease: DISEASE_DISEASE con grupos BERT", "Disease", pares("DISEASE_DISEASE")),
    metricas_plano("disease: DISEASE_DISEASE sin grupos BERT", "Disease", pares("DISEASE_DISEASE", sin_grupos=True)),
    metricas_plano("drug: DRUG_DRUG", "Drug", pares("DRUG_DRUG")),
]).set_index("plano")
planos

similaridad minima en DRUG_DRUG: 0.81


,nodos con arista,fraccion de la capa,aristas,componentes,LCC,LCC / nodos,kappa,grado mediano
plano,,,,,,,,
gene: PPI,13894,0.802,110051,34,13819,0.995,60.5,8.0
disease: DISEASE_DISEASE con grupos BERT,7286,0.484,11456,130,6844,0.939,9.3,2.0
disease: DISEASE_DISEASE sin grupos BERT,5318,0.354,7836,185,4644,0.873,7.4,2.0
drug: DRUG_DRUG,36632,0.437,72557,9485,305,0.008,13.8,2.0


**Lectura.** Los planos de genes y enfermedades son los de PrimeKG integrado: el PPI tiene 13.894 nodos con componente gigante casi total, y el plano de enfermedades sin grupos queda en 5.318 nodos y 7.836 aristas. El de drogas es otra cosa: 36.632 compuestos con arista repartidos en 9.485 componentes, con la mayor en 305 nodos (menos del 1 %). La kappa de 13,8 no indica componente gigante acá: sale de los clusters del laboratorio, que se pasan a todos los pares de compuestos y arman grupos densos y aislados entre sí. Es la fragmentación esperable de una similaridad química, que agrupa familias de compuestos.

## 5. Cobertura de las relaciones entre capas

Para cada relación intercapa, cuántos nodos de cada lado tienen al menos una arista, qué fracción de su capa representan y el grado mediano y máximo. Una relación cargada al revés o contra el label equivocado se vería acá como un lado vacío o con un grado absurdo.

In [7]:
filas = []
for tipo in ["GDA", "INDICATION", "DRUG_TARGET", "PATHWAY_PROTEIN", "FORM_COMPLEX", "DISEASE_BERT"]:
    sub = aristas[aristas.tipo == tipo]
    for lado, label in [("id_a", sub.label_a.iloc[0]), ("id_b", sub.label_b.iloc[0])]:
        grados = sub[lado].value_counts()
        total = int((nodos.label == label).sum())
        filas.append({"relacion": tipo, "lado": label, "nodos con arista": len(grados), "de la capa": total,
                      "fraccion": round(len(grados) / total, 3), "grado mediano": float(grados.median()), "grado maximo": int(grados.max()),
                      "nodo de grado maximo": nodos.set_index(["label", "id"]).name.get((label, grados.index[0]), grados.index[0])})
pd.DataFrame(filas)

,relacion,lado,nodos con arista,de la capa,fraccion,grado mediano,grado maximo,nodo de grado maximo
0,GDA,Gene,9689,17743,0.546,4.0,340,TNF
1,GDA,Disease,11166,16079,0.694,1.0,1074,Malignant neoplasm of breast
2,INDICATION,Drug,1458,83867,0.017,3.0,152,Prednisolone
3,INDICATION,Disease,1175,16079,0.073,3.0,101,hypertension
4,DRUG_TARGET,Drug,82923,83867,0.989,1.0,264,CHEMBL14830
5,DRUG_TARGET,Gene,3258,17743,0.184,3.0,15658,BAZ2B
6,PATHWAY_PROTEIN,Gene,10846,17743,0.611,2.0,191,RPS27A
7,PATHWAY_PROTEIN,Pathway,2017,2017,1.000,10.0,478,Neutrophil degranulation
8,FORM_COMPLEX,Gene,1497,17743,0.084,1.0,12,ITGB1
9,FORM_COMPLEX,Gene,421,17743,0.024,2.0,53,60S cytosolic large ribosomal subunit


**Lectura.** Los extremos tienen el tipo que corresponde a cada fuente y no hay lados vacíos. Las dos relaciones de drogas tienen coberturas opuestas: el 98,9 % de los compuestos tiene alguna diana, porque así entran a D\*, y solo el 1,7 % (1.458) tiene alguna indicación. Del otro lado, las indicaciones tocan el 7 % de las enfermedades y las dianas el 18 % de los genes. El grado de BAZ2B en `DRUG_TARGET` (15.658 compuestos) es el de una diana de cribado masivo: no es un error de carga, pero es un hub que viene del diseño del ensayo y no de la biología.

## 6. Tres enfermedades de ejemplo

Esquizofrenia, celiaquía y gripe. Para cada una se buscan sus nodos por nombre y se muestra con qué está conectada: su jerarquía de MONDO (`DISEASE_DISEASE`), la estrella del grupo BERT si pertenece a uno (`DISEASE_BERT`), sus genes (`GDA`) y sus drogas (`INDICATION`). Esquizofrenia muestra el reparto que hace PrimeKG integrado: el CUI de DisGeNET lleva los genes y el grupo BERT lleva la jerarquía y las drogas, unidos solo por la estrella.

In [8]:
nombre = nodos.set_index("id").name.to_dict()
EJEMPLOS = {"esquizofrenia": ["C0036341"], "celiaquia": ["C0007570"], "gripe": ["C0021400"]}

#el grupo BERT de cada CUI de ejemplo, si tiene uno: se suma al ejemplo porque lleva la otra mitad de sus aristas
estrella_neo4j = aristas[aristas.tipo == "DISEASE_BERT"]
for clave, ids in EJEMPLOS.items():
    grupo = estrella_neo4j[estrella_neo4j.id_a.isin(ids) | estrella_neo4j.id_b.isin(ids)]
    ids += sorted(set(grupo.id_a) | set(grupo.id_b) - set(ids))


def vecinos(ids, tipo, label_vecino):
    sub = aristas[(aristas.tipo == tipo) & (aristas.id_a.isin(ids) | aristas.id_b.isin(ids))]
    otros = [b if a in ids else a for a, b in zip(sub.id_a, sub.id_b)]
    labels = [lb if a in ids else la for a, la, lb in zip(sub.id_a, sub.label_a, sub.label_b)]
    return sorted({o for o, l in zip(otros, labels) if l == label_vecino and o not in ids})


resumen_ejemplos = []
for clave, ids in EJEMPLOS.items():
    for i in ids:
        resumen_ejemplos.append({"ejemplo": clave, "nodo": i[:40], "nombre": nombre[i],
                                 "genes (GDA)": len(vecinos([i], "GDA", "Gene")), "drogas (INDICATION)": len(vecinos([i], "INDICATION", "Drug")),
                                 "enfermedades (DISEASE_DISEASE)": len(vecinos([i], "DISEASE_DISEASE", "Disease")),
                                 "estrella (DISEASE_BERT)": len(vecinos([i], "DISEASE_BERT", "Disease"))})
pd.DataFrame(resumen_ejemplos)

,ejemplo,nodo,nombre,genes (GDA),drogas (INDICATION),enfermedades (DISEASE_DISEASE),estrella (DISEASE_BERT)
0,esquizofrenia,C0036341,Schizophrenia,883,0,0,1
1,esquizofrenia,5090_13498_8414_10897_33312_10943_11552_,schizophrenia,0,35,5,14
2,celiaquia,C0007570,Celiac Disease,47,0,3,0
3,gripe,C0021400,Influenza,52,8,3,0


In [9]:
for clave, ids in EJEMPLOS.items():
    print(f"== {clave}")
    print("  enfermedades vecinas por MONDO:", [nombre[v] for v in vecinos(ids, "DISEASE_DISEASE", "Disease")])
    drogas = vecinos(ids, "INDICATION", "Drug")
    print(f"  drogas con indicacion ({len(drogas)}):", [nombre[v] for v in drogas][:25])
    genes = vecinos(ids, "GDA", "Gene")
    print(f"  genes ({len(genes)}), algunos:", [nombre[v] for v in genes][:15])

== esquizofrenia
  enfermedades vecinas por MONDO: ['myopia', 'Other and unspecified reactive psychosis', 'Psychotic Disorders', 'Paranoid Schizophrenia', 'CHROMOSOME 2p16.3 DELETION SYNDROME']
  drogas con indicacion (35): ['Chlorprothixene', 'Flupentixol', 'Trifluoperazine', 'Asenapine', 'Lumateperone', 'Sertindole', 'Fluspirilene', 'Mesoridazine', 'Haloperidol', 'Perphenazine', 'Olanzapine', 'Risperidone', 'Iloperidone', 'Zuclopenthixol', 'Perospirone', 'Aripiprazole', 'Paliperidone', 'Perazine', 'Chlorpromazine', 'Blonanserin', 'Cariprazine', 'Molindone', 'Promazine', 'Prochlorperazine', 'Fluphenazine']
  genes (883), algunos: ['A1BG', 'LINC00271', 'PLA2G4B', 'DEL22Q11.2', 'DLG2-AS1', 'TSNAX-DISC1', 'RANBP9', 'MIR2682', 'ERVW-4', 'ACTR2', 'HNRNPA3P1', 'CDK5', 'OLIG2', 'PRSS16', 'CDKN1C']
== celiaquia
  enfermedades vecinas por MONDO: ['celiac disease, susceptibility to', 'Malabsorption Syndrome', 'Disorder of small intestine']
  drogas con indicacion (0): []
  genes (47), algunos: 

**Lectura.** Las tres enfermedades aparecen con conexiones que tienen sentido clínico. La esquizofrenia muestra el reparto de PrimeKG integrado: el CUI tiene los 883 genes y ninguna droga, y su grupo BERT tiene las 35 drogas (antipsicóticos, típicos y atípicos) y la jerarquía de MONDO, unidos por la estrella. Entre sus vecinos de MONDO aparece miopía: es una arista padre-hijo que ya está en PrimeKG nativo entre dos nodos agrupados, no un error de carga. La celiaquía no tiene drogas, y es correcto: no hay fármaco aprobado, el tratamiento es la dieta. Sus genes incluyen HLA-DQA1 y HLA-DQB1, el locus de riesgo conocido. La gripe tiene ocho antivirales (cuatro inhibidores de neuraminidasa, baloxavir, favipiravir, umifenovir y moroxidina), y sus genes son de respuesta a interferón (IFITM3, IFI44, DDX58).

### Drogas y genes de una misma enfermedad

Si la carga respeta la biología, las drogas indicadas para una enfermedad deberían apuntar, al menos a veces, a genes asociados a esa enfermedad. Para cada ejemplo se cruzan las dianas (`DRUG_TARGET`) de sus drogas con sus genes (`GDA`).

In [10]:
dianas_por_droga = defaultdict(set)
for a, b in aristas[aristas.tipo == "DRUG_TARGET"][["id_a", "id_b"]].itertuples(index=False):
    dianas_por_droga[a].add(b)

for clave, ids in EJEMPLOS.items():
    genes = set(vecinos(ids, "GDA", "Gene"))
    drogas = vecinos(ids, "INDICATION", "Drug")
    con_diana = [d for d in drogas if dianas_por_droga[d]]
    en_comun = {d: sorted(nombre[g] for g in dianas_por_droga[d] & genes) for d in con_diana if dianas_por_droga[d] & genes}
    print(f"== {clave}: {len(drogas)} drogas, {len(con_diana)} con alguna diana, {len(en_comun)} con una diana entre los {len(genes)} genes de la enfermedad")
    for d, gs in en_comun.items():
        print(f"   {nombre[d]}: {gs}")

== esquizofrenia: 35 drogas, 16 con alguna diana, 7 con una diana entre los 883 genes de la enfermedad
   Lumateperone: ['HTR2A']
   Fluspirilene: ['NPSR1', 'XBP1']
   Haloperidol: ['DRD2', 'SIGMAR1']
   Risperidone: ['HTR2A']
   Aripiprazole: ['HTR2A']
   Cariprazine: ['HRH1', 'HTR2A']
   Brexpiprazole: ['HRH1']
== celiaquia: 0 drogas, 0 con alguna diana, 0 con una diana entre los 47 genes de la enfermedad
== gripe: 8 drogas, 3 con alguna diana, 0 con una diana entre los 52 genes de la enfermedad


**Lectura.** En la esquizofrenia, 7 de las 16 drogas con diana apuntan a un gen de la enfermedad, y son las dianas farmacológicas de manual: DRD2 para el haloperidol y HTR2A para los atípicos. En la gripe ninguna de las tres con diana coincide, y también es lo esperable: los antivirales actúan sobre proteínas del virus, que no son humanas y quedaron afuera del mapeo; las dianas que sí tienen son proteínas humanas medidas en ensayos.

### Cómo se conectan los tres ejemplos entre sí

Genes compartidos entre cada par de ejemplos, y el camino más corto entre ellos en dos grafos: el plano de enfermedades (solo `DISEASE_DISEASE` y la estrella) y el grafo entero.

In [11]:
from itertools import combinations

genes_de = {clave: set(vecinos(ids, "GDA", "Gene")) for clave, ids in EJEMPLOS.items()}
for a, b in combinations(EJEMPLOS, 2):
    comunes = sorted(nombre[g] for g in genes_de[a] & genes_de[b])
    print(f"{a} y {b}: {len(comunes)} genes en comun {comunes[:20]}")

plano_enfermedades = nx.Graph()
plano_enfermedades.add_edges_from(pares("DISEASE_DISEASE") + pares("DISEASE_BERT"))
grafo_entero = nx.Graph()
grafo_entero.add_edges_from(zip(aristas.label_a + ":" + aristas.id_a, aristas.label_b + ":" + aristas.id_b))

print()
for a, b in combinations(EJEMPLOS, 2):
    origen_a, destino_b = EJEMPLOS[a][0], EJEMPLOS[b][0]
    try:
        camino = nx.shortest_path(plano_enfermedades, origen_a, destino_b)
        print(f"plano de enfermedades, {a} a {b}: {len(camino) - 1} saltos:", " > ".join(nombre[n] for n in camino))
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        print(f"plano de enfermedades, {a} a {b}: sin camino")
    camino = nx.shortest_path(grafo_entero, "Disease:" + origen_a, "Disease:" + destino_b)
    print(f"grafo entero, {a} a {b}: {len(camino) - 1} saltos:", " > ".join(f"{nombre[n.split(':', 1)[1]]} ({n.split(':')[0]})" for n in camino))

esquizofrenia y celiaquia: 12 genes en comun ['ACE', 'CTLA4', 'HLA-DQB1', 'IFNG', 'LCN2', 'LPL', 'MAGI2', 'MMP3', 'MYO9B', 'PCK1', 'S100A9', 'TGM2']
esquizofrenia y gripe: 1 genes en comun ['CCL2']
celiaquia y gripe: 2 genes en comun ['CXCL10', 'IFI27']

plano de enfermedades, esquizofrenia a celiaquia: 8 saltos: Schizophrenia > schizophrenia > Psychotic Disorders > Mental disorders > anxiety disorder > Nutrition Disorders > lactose intolerance > Malabsorption Syndrome > Celiac Disease
grafo entero, esquizofrenia a celiaquia: 2 saltos: Schizophrenia (Disease) > HLA-DQB1 (Gene) > Celiac Disease (Disease)
plano de enfermedades, esquizofrenia a gripe: 8 saltos: Schizophrenia > schizophrenia > myopia > Refractive Errors > Disorder of eye > Eye Infection > infectious disease > Orthomyxoviridae Infections > Influenza
grafo entero, esquizofrenia a gripe: 2 saltos: Schizophrenia (Disease) > CCL2 (Gene) > Influenza (Disease)
plano de enfermedades, celiaquia a gripe: 6 saltos: Celiac Disease > M

**Lectura.** Por genes, esquizofrenia y celiaquía comparten 12 (HLA-DQB1, CTLA4, TGM2 entre ellos), y los caminos más cortos en el grafo entero pasan por un gen, en dos saltos. En el plano de enfermedades solo, en cambio, las tres quedan a seis u ocho saltos subiendo por categorías amplias de MONDO (trastornos mentales, enfermedades infecciosas). Es la diferencia que el algoritmo aprovecha: la ontología separa lo que la biología molecular acerca.

## 7. Dos pruebas de sentido biológico sobre toda la capa de drogas

Las métricas de arriba dicen que la carga es coherente con los archivos. Estas dos dicen si lo que se cargó tiene sentido como biología, comparando contra un nulo con la semilla 42.

**Indicaciones explicadas por dianas.** Para cada arista `INDICATION` (droga, enfermedad), ¿alguna diana de la droga es un gen de la enfermedad? Para los grupos BERT, que no tienen genes propios, se usan los genes de sus miembros. El nulo reasigna al azar las enfermedades entre las indicaciones de drogas con alguna diana, conservando cuántas indicaciones tiene cada droga y cada enfermedad.

**Drogas parecidas comparten indicaciones.** Para los pares unidos por `DRUG_DRUG`, el Jaccard entre sus conjuntos de enfermedades, por tramo de similaridad, contra pares de drogas al azar. Se miden solo los compuestos con alguna indicación, de los dos lados: sin indicaciones el Jaccard es 0 por construcción, y con el 98 % de los compuestos en ese caso la prueba no mediría nada.

In [12]:
rng = np.random.default_rng(SEED)
genes_por_enfermedad = defaultdict(set)
for a, la, b in aristas[aristas.tipo == "GDA"][["id_a", "label_a", "id_b"]].itertuples(index=False):
    #GDA puede estar guardada de Gene a Disease o al reves, el label de cada extremo dice cual es cual
    gen, enfermedad = (a, b) if la == "Gene" else (b, a)
    genes_por_enfermedad[enfermedad].add(gen)
for a, b in estrella_neo4j[["id_a", "id_b"]].itertuples(index=False):
    grupo, miembro = (a, b) if a in grupos_bert else (b, a)
    genes_por_enfermedad[grupo] |= genes_por_enfermedad[miembro]

indicaciones = aristas[aristas.tipo == "INDICATION"][["id_a", "id_b"]].rename(columns={"id_a": "droga", "id_b": "enfermedad"})
indicaciones = indicaciones[indicaciones.droga.map(lambda d: len(dianas_por_droga[d]) > 0)]


def fraccion_explicada(tabla):
    return float(np.mean([len(dianas_por_droga[d] & genes_por_enfermedad[e]) > 0 for d, e in zip(tabla.droga, tabla.enfermedad)]))


observada = fraccion_explicada(indicaciones)
nulos = [fraccion_explicada(indicaciones.assign(enfermedad=rng.permutation(indicaciones.enfermedad.values))) for _ in range(200)]
print(f"indicaciones de drogas con diana: {len(indicaciones)}")
print(f"fraccion explicada por una diana: observada {observada:.3f}, nulo {np.mean(nulos):.3f} +- {np.std(nulos):.3f}, "
      f"z = {(observada - np.mean(nulos)) / np.std(nulos):.1f}")

indicaciones de drogas con diana: 3432
fraccion explicada por una diana: observada 0.065, nulo 0.023 +- 0.002, z = 17.6


**Lectura.** El 6,5 % de las 3.432 indicaciones de drogas con diana tiene una diana entre los genes de la enfermedad, contra 2,3 % en el nulo: casi tres veces más, a 17,6 desvíos. La fracción absoluta es baja porque las dianas son bioactividad medida y no dianas curadas, y porque muchas enfermedades tienen pocos genes, pero la señal está y va en el sentido correcto. Si las indicaciones o las dianas se hubieran pegado a nodos equivocados, esta diferencia desaparecería.

In [13]:
enfermedades_por_droga = defaultdict(set)
for d, e in aristas[aristas.tipo == "INDICATION"][["id_a", "id_b"]].itertuples(index=False):
    enfermedades_por_droga[d].add(e)


def jaccard(a, b):
    union = enfermedades_por_droga[a] | enfermedades_por_droga[b]
    return len(enfermedades_por_droga[a] & enfermedades_por_droga[b]) / len(union) if union else 0.0


#solo los compuestos con alguna indicacion: sin indicaciones el Jaccard es 0 por construccion
con_indicacion = set(enfermedades_por_droga)
similares = aristas[aristas.tipo == "DRUG_DRUG"][["id_a", "id_b", "weight"]]
similares = similares[similares.id_a.isin(con_indicacion) & similares.id_b.isin(con_indicacion)].copy()
print("compuestos con alguna indicacion:", len(con_indicacion), "| pares DRUG_DRUG entre ellos:", len(similares))
similares["jaccard"] = [jaccard(a, b) for a, b in zip(similares.id_a, similares.id_b)]
similares["tramo"] = pd.cut(similares.weight, [0.8, 0.9, 1.0001], right=False, labels=["0,81 a 0,9", "0,9 a 1"])
drogas_ids = sorted(con_indicacion)
azar = rng.choice(len(drogas_ids), size=(5000, 2))
azar = [(drogas_ids[i], drogas_ids[j]) for i, j in azar if i != j]
tabla_jaccard = similares.groupby("tramo", observed=True).jaccard.agg(["size", "mean", lambda s: (s > 0).mean()])
tabla_jaccard.columns = ["pares", "jaccard medio", "fraccion con alguna enfermedad en comun"]
tabla_jaccard.loc["pares al azar"] = [len(azar), np.mean([jaccard(a, b) for a, b in azar]), np.mean([jaccard(a, b) > 0 for a, b in azar])]
tabla_jaccard.round(3)

compuestos con alguna indicacion: 1458 | pares DRUG_DRUG entre ellos: 249


,pares,jaccard medio,fraccion con alguna enfermedad en comun
tramo,,,
"0,81 a 0,9",159.0,0.325,0.667
"0,9 a 1",90.0,0.334,0.600
pares al azar,4995.0,0.008,0.023


**Lectura.** Entre los 1.458 compuestos con indicación hay 249 pares unidos por similaridad. Dos compuestos parecidos comparten indicaciones mucho más que dos al azar: el 67 % de los pares entre 0,81 y 0,9 y el 60 % de los de 0,9 o más tienen alguna enfermedad en común, contra el 2,3 % al azar. Entre los dos tramos no hay gradiente, pero con 159 y 90 pares la diferencia no dice nada. Es coherente con que el plano droga-droga y las indicaciones vienen de fuentes independientes (la red del laboratorio y PrimeKG) y se cargaron sobre los mismos nodos.

In [14]:
#los nodos de mayor grado por capa, para detectar hubs artificiales
grado = Counter(list(aristas.label_a + ":" + aristas.id_a) + list(aristas.label_b + ":" + aristas.id_b))
filas = []
for capa in CAPAS:
    top = sorted(((g, n) for n, g in grado.items() if n.startswith(capa + ":")), reverse=True)[:5]
    for g, n in top:
        filas.append({"capa": capa, "nodo": nombre[n.split(":", 1)[1]], "grado": g})
pd.DataFrame(filas)

,capa,nodo,grado
0,Gene,BAZ2B,15661
1,Gene,ATAD5,11927
2,Gene,RAB9A,9171
3,Gene,NPC1,7643
4,Gene,HPGD,6419
5,Disease,Malignant neoplasm of breast,1082
6,Disease,Schizophrenia,884
7,Disease,"Liver Cirrhosis, Experimental",774
8,Disease,Colorectal Carcinoma,707
9,Disease,Malignant neoplasm of prostate,627


**Lectura.** En enfermedades y pathways los hubs son los esperables (los cánceres frecuentes, la esquizofrenia por sus 883 genes, la degranulación de neutrófilos). En genes, en cambio, los cinco primeros (BAZ2B, ATAD5, RAB9A, NPC1, HPGD) no son los del interactoma sino dianas de cribado masivo: su grado sale casi entero de `DRUG_TARGET`. En drogas, los tres primeros son compuestos sin nombre, identificados por su ChEMBL, con muchas actividades medidas, y después vienen los corticoides por sus indicaciones. Ningún hub es un id mal resuelto, pero los de genes y drogas reflejan cómo se diseñaron los ensayos de bioactividad.

In [15]:
driver.close()

## Conclusión: la carga funcionó

Tres niveles de evidencia, de lo formal a lo biológico:

1. **Fidelidad con los archivos.** Los nodos por subtipo y las aristas por tipo coinciden uno a uno con los CSV de origen (119.706 nodos, 468.052 aristas), sin ids repetidos, sin relaciones duplicadas en ningún sentido y sin bucles. Lo de PrimeKG integrado entra intacto. `neo4j/cargar_grafo_laboratorio.py` comprueba lo mismo al final de la carga, con código propio.
2. **Estructura coherente con el diseño.** Cada relación une siempre el mismo par de capas, las combinaciones que el diseño no conecta están en cero, y los planos de genes y enfermedades tienen los tamaños de PrimeKG integrado. El plano de drogas está muy fragmentado, pero eso es una propiedad de la similaridad química, no de la carga.
3. **Sentido biológico.** Los ejemplos se leen bien (antipsicóticos para la esquizofrenia con DRD2 y HTR2A entre sus dianas, celiaquía sin fármacos y con HLA-DQ, antivirales para la gripe), las indicaciones se explican por dianas casi tres veces más que al azar, y los compuestos químicamente parecidos comparten indicaciones mucho más que los elegidos al azar. Estas dos pruebas unen fuentes que se armaron por separado, y solo dan señal si los tres cruces (DrugBank a PrimeKG, PrimeKG a CUI, UniProt a Entrez) pusieron cada arista sobre el nodo correcto.

Lo que no es error de carga pero conviene tener presente: la esquizofrenia y otras enfermedades con grupo BERT reparten genes y drogas entre dos nodos, así que excluir los grupos deja sin indicaciones a sus CUIs; la arista esquizofrenia a miopía viene de PrimeKG nativo; las dianas son bioactividad medida, lo que explica que la fracción de indicaciones explicadas sea baja en términos absolutos; y los ensayos de cribado masivo dejan hubs artificiales del lado de los genes (BAZ2B con 15.658 compuestos).